# Policy dispatcher run

This notebook runs a config through the dispatcher, writes the Zarr result store under `artifacts/notebook_runs/`, and then reads the generated files back.

## Imports

In [ ]:
import json
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dask.distributed import Client, LocalCluster

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tasks.dispatch.dispatcher import dispatch_from_config
from tasks.policies import load_policy_results

## Start a local Dask cluster

In [ ]:
n_workers = 4
threads_per_worker = 1

cluster = LocalCluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    processes=False,
    dashboard_address=None,
)
client = Client(cluster)

{
    "scheduler_address": cluster.scheduler_address,
    "dashboard_link": client.dashboard_link,
    "n_workers": len(client.scheduler_info()["workers"]),
    "threads_per_worker": threads_per_worker,
}


## Load the checked-in config and prepare runtime settings

In [ ]:
source_config_path = project_root / "configs" / "best_response_policy" / "baseline.dispatch_config.json"
source_config = json.loads(source_config_path.read_text(encoding="utf-8"))

started_slug = datetime.now().astimezone().strftime("%Y%m%dT%H%M%S")
run_dir = project_root / "artifacts" / "notebook_runs" / f"best-response-policy-{started_slug}"
run_dir.mkdir(parents=True, exist_ok=False)

runtime_config = dict(source_config)
runtime_config["scheduler_address"] = cluster.scheduler_address
runtime_config["output_dir"] = str(run_dir)

{
    "source_config_path": str(source_config_path),
    "run_dir": str(run_dir),
    "scheduler_address": runtime_config["scheduler_address"],
    "output_dir": runtime_config["output_dir"],
    "write_batch_size": runtime_config["write_batch_size"],
}


## Dispatch the task

In [ ]:
with tempfile.NamedTemporaryFile(
    mode="w",
    suffix=".dispatch_config.json",
    encoding="utf-8",
    delete=False,
) as handle:
    json.dump(runtime_config, handle, indent=2)
    handle.write("\n")
    temp_config_path = Path(handle.name)

try:
    dispatch_summary = dispatch_from_config(str(temp_config_path))
finally:
    temp_config_path.unlink(missing_ok=True)

dispatch_summary


## Load formatted policy results

In [ ]:
result_store_path = Path(dispatch_summary["output_zarr"])
policy_results = load_policy_results(result_store_path)
input_index = policy_results.metadata["input_index"]
parameter_values = policy_results.inputs["tensors"]["parameters"]
output_tensors = policy_results.outputs["tensors"]
output_labels = policy_results.outputs["labels"]

{
    "result_store_path": str(policy_results.metadata["result_store_path"]),
    "task_name": policy_results.metadata["task_name"],
    "task_class": policy_results.metadata["task_class"],
    "average_completion_duration_ms": policy_results.metadata["average_completion_duration_ms"],
    "stored_results": int(input_index.shape[0]),
    "stored_failures": int(policy_results.failures["error"].shape[0]) if "error" in policy_results.failures else 0,
    "parameters_shape": tuple(parameter_values.shape),
    "output_tensors": {name: tuple(values.shape) for name, values in output_tensors.items()},
    "run_dir_files": sorted(path.name for path in run_dir.iterdir()),
}


In [ ]:
timeseries = output_tensors["timeseries"]
observables = output_labels["observable"]
steps = output_labels["step"]

fig, axes = plt.subplots(1, len(observables), figsize=(12, 4), sharex=True)
axes = np.atleast_1d(axes)
for observable_index, observable_name in enumerate(observables):
    for row_index, sample_idx in enumerate(input_index):
        axes[observable_index].plot(
            steps,
            timeseries[row_index, observable_index],
            alpha=0.8,
            label=f"sample {int(sample_idx)}",
        )
    axes[observable_index].set_title(str(observable_name))
    axes[observable_index].set_xlabel("step")
    axes[observable_index].set_ylabel("value")
    axes[observable_index].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
summary = {"sample": input_index}
for parameter_index, parameter_name in enumerate(policy_results.inputs["labels"]["parameter"]):
    summary[f"parameter.{parameter_name}"] = parameter_values[:, parameter_index]

for observable_index, observable_name in enumerate(observables):
    summary[f"{observable_name}.final"] = output_tensors["final"][:, observable_index]

summary


## Cleanup

In [ ]:
client.close()
cluster.close()
